In [1]:
import os

import random

import numpy as np

import cv2

import matplotlib.pyplot as plt
import seaborn as sns

from keras.applications import DenseNet121
from keras.models import Sequential
from keras import layers
from keras.callbacks import ReduceLROnPlateau

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tqdm.notebook import tqdm

from src.module.ai_module import apply_preprocessing

In [2]:
path = "dataset"

In [3]:
os.chdir("..")
os.chdir("..")
os.chdir(path)

In [4]:
labels = ["0","1","2","3","4"]
img_size = 96

In [5]:
def get_data(data_dir):  # train setindeki hasta sayısı ile sınırlandır
    
    data = list()
    
    for label in labels:
        class_id = labels.index(label) # 'NORMAL' = 0, 'STROKE' = 1
        main_dir = os.path.join(os.getcwd(), data_dir)
        main_dir = os.path.join(main_dir, label)
              
        for img_name in tqdm(os.listdir(main_dir)):
            
            
            try:
                # read image
                img = cv2.imread(os.path.join(main_dir, img_name), cv2.IMREAD_COLOR)
                
                
                # apply preprocessing
                img = apply_preprocessing(img,img_size)
                
                data.append([img, class_id])
                        
            
            except Exception as e:
                print(e)
    
    return data 

In [ ]:
train = get_data("TRAIN")

In [ ]:
test = get_data("TEST")

In [8]:
random.shuffle(train)
random.shuffle(test)

In [9]:
print(len(train), len(test))

5684 960


In [10]:
train = np.array(train, dtype = 'object')    
test = np.array(test, dtype = 'object')    

In [11]:
x_train, y_train = list(), list()
x_test, y_test = list(), list()

for feat, label in train:
    x_train.append(feat) # görseller
    y_train.append(label) # etiketler

In [12]:
for feat, label in test:
    x_test.append(feat)
    y_test.append(label)

In [13]:
# normalization : [0,255] -> [0,1]

x_train = np.array(x_train) / 255
x_test = np.array(x_test) / 255

In [14]:
x_train = x_train.reshape(-1, img_size, img_size, 3) # -1 : oto 
x_test = x_test.reshape(-1, img_size, img_size, 3)

In [15]:
y_train = np.array(y_train)
y_test = np.array(y_test)

In [47]:
model_name = "DenseNet121"

model = Sequential()
model.add(DenseNet121(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3)))
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dropout(0.3))  # Daha az dropout
model.add(layers.Dense(5, activation='softmax'))  # sigmoid yerine softmax


In [48]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [49]:
lr = ReduceLROnPlateau(monitor = 'val_loss',
                      patience = 3,
                      verbose = 1,
                      mode='auto',
                      factor=0.25,
                      min_lr=1e-6)

In [ ]:
history = model.fit(x_train, y_train,
                    batch_size=4,
                    epochs=19,
                    validation_data=(x_test, y_test),
                    callbacks=[lr])
    

In [51]:
# History'den eğitim ve doğrulama metriklerini alma
history_dict = history.history

In [52]:
# Eğitim ve doğrulama kaybı (loss)
loss_values = history_dict['loss']
val_loss_values = history_dict['val_loss']

In [53]:
# Eğitim ve doğrulama doğruluğu (accuracy)
acc_values = history_dict['accuracy']
val_acc_values = history_dict['val_accuracy']

In [54]:
epochs = range(1, len(loss_values) + 1)

In [59]:
# Modelin tahminleri
y_pred_probs = model.predict(x_test)  # softmax çıktıları
y_pred = np.argmax(y_pred_probs, axis=1)  # en yüksek olasılığı al


30/30 [==============================] - 1s 34ms/step


In [ ]:
cm = confusion_matrix(y_test, y_pred)

# Isı haritası çiz
plt.figure(figsize=(8, 6))  # Daha büyük bir figür boyutu
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", linewidths=1.5, linecolor='black', 
            xticklabels=set(y_test), yticklabels=set(y_test))

plt.title(f"Orijinal Test Verisi - {model_name}", fontsize=14, pad=15)
plt.show()

In [61]:
print("accuracy : \n",accuracy_score(y_test, y_pred))
print("cr : \n",classification_report(y_test, y_pred))   

accuracy : 
 0.6125
cr : 
               precision    recall  f1-score   support

           0       0.81      0.92      0.86       192
           1       0.60      0.67      0.63       192
           2       0.44      0.58      0.50       192
           3       0.64      0.48      0.55       192
           4       0.60      0.41      0.49       192

    accuracy                           0.61       960
   macro avg       0.62      0.61      0.61       960
weighted avg       0.62      0.61      0.61       960



In [27]:
os.chdir("..")


In [58]:
import json
import numpy as np
from tensorflow.keras.models import model_from_config

def fix_json_serialization(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if hasattr(value, "numpy"):  # Tensor ise
        return value.numpy().tolist()
    return value

# Model konfigürasyonunu al
config = model.get_config()

# JSON'a uygun hale getir
fixed_config = json.loads(json.dumps(config, default=fix_json_serialization))

# Sequential modelse model_from_config ile tekrar yükle
from tensorflow.keras.models import Sequential
fixed_model = Sequential.from_config(fixed_config)

# Ağırlıkları aktar
fixed_model.set_weights(model.get_weights())

# Kaydet
fixed_model.save(f"models/{model_name}.h5")